# ICBHI AST-SAM — Kaggle Training Notebook

**Required Settings (Right Sidebar):**
1. **Internet**: ON (to clone GitHub and install packages).
2. **Persistence**: Filesystem (to keep your checkpoints).
3. **Accelerator**: GPU P100 (Recommended) or T4 x2.

In [1]:
# 1. Setup paths and verify GPU
import os
import torch

ROOT_DIR = '/kaggle/working'
REPO_DIR = os.path.join(ROOT_DIR, 'SAM-Optimized-AST-Respiratory-Sound-Classification')
os.chdir(ROOT_DIR)

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [2]:
# 2. Clone Repository
REPO_URL = 'https://github.com/ajammoussi/SAM-Optimized-AST-Respiratory-Sound-Classification'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR
!git pull

Cloning into 'SAM-Optimized-AST-Respiratory-Sound-Classification'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 110 (delta 57), reused 86 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 1.94 MiB | 9.05 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/kaggle/working/SAM-Optimized-AST-Respiratory-Sound-Classification
Already up to date.


In [3]:
# 3. Install dependencies
!pip install -r requirements.txt -q
!pip install gdown -q

In [4]:
# 4. Dataset Setup
import os
NPZ_FILE = 'icbhi_ast_16k_8s_spectrograms.npz'

# If it's not already there, download it from Drive
if not os.path.exists(NPZ_FILE):
    print("Downloading dataset from Google Drive...")
    # REPLACE THE ID BELOW WITH YOUR ACTUAL FILE ID
    FILE_ID = '1oZsA_rmUpaijwQEWeesUfsMgcmj7OIiB' 
    !gdown --id $FILE_ID -O $NPZ_FILE
else:
    print("✓ Dataset found locally.")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1oZsA_rmUpaijwQEWeesUfsMgcmj7OIiB
From (redirected): https://drive.google.com/uc?id=1oZsA_rmUpaijwQEWeesUfsMgcmj7OIiB&confirm=t&uuid=7204e9db-c07f-4305-bc98-5c3e495074bb
To: /kaggle/working/SAM-Optimized-AST-Respiratory-Sound-Classification/icbhi_ast_16k_8s_spectrograms.npz
100%|███████████████████████████████████████| 2.28G/2.28G [00:17<00:00, 130MB/s]


In [5]:
# 5. Training
# Note: Remove --resume for the very first run
!python scripts/train.py \
    --data_path    ./icbhi_ast_16k_8s_spectrograms.npz \
    --checkpoint_dir ./checkpoints \
    --results_dir  ./results \
    --epochs       20 \
    --batch_size   24 \
    --lr           1e-5 \
    --rho          0.05 \
    --looksam_k    2 \
    --num_workers  2 \
    --scheduler_t0   10 \
    --amp_dtype    fp16 \
    --min_se       0.6 \
    --min_sp       0.6 \
    #--resume

Device : cuda  |  autocast : True (fp16)  |  GradScaler : True

Loading data from: ./icbhi_ast_16k_8s_spectrograms.npz
   Train: torch.Size([4131, 1024, 128])   Test: torch.Size([2756, 1024, 128])
   Class counts: {0: np.int64(2057), 1: np.int64(1210), 2: np.int64(501), 3: np.int64(363)}

Building model …
config.json: 26.8kB [00:00, 37.9MB/s]
model.safetensors:  77%|█████████████████     | 268M/346M [00:02<00:00, 127MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1580.42it/s, Materializing param=
ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Training starts at epo

In [6]:
# 6. Evaluation
!python scripts/evaluate.py \
    --data_path  ./icbhi_ast_16k_8s_spectrograms.npz \
    --model_path ./checkpoints/best_model.pth \
    --output_dir ./results

  Device: cuda  |  AMP: True

 Loading data: ./icbhi_ast_16k_8s_spectrograms.npz
 Loading model: ./checkpoints/best_model.pth
Loading weights: 100%|█| 199/199 [00:00<00:00, 1437.96it/s, Materializing param=
ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[OK] Model loaded successfully.

 Running inference …

  ICBHI 2017 Evaluation Results
  Sensitivity (Se) : 61.77%
  Specificity (Sp) : 75.49%
  ICBHI Score      : 68.63%

Per-class report:
              precision    recall  f1-score   support

      Normal       0.73      0.75      0.74      1579
     Crackle       0.57 